In [4]:
import json
import numpy as np
from PIL import Image
import tifffile
import zarr
from shapely.geometry import shape, Polygon, MultiPolygon
from rasterio import features, Affine
import os
from tqdm import tqdm
from pathlib import Path

In [5]:


class QuPathTiffExtractor:
    def __init__(self, tif_path, geojson_path, tile_size=512, overlap=256):
        self.tif_path = tif_path
        self.tile_size = tile_size
        self.overlap = overlap
        
        self.tiff = tifffile.TiffFile(tif_path)
        self._init_image_properties()
        
        with open(geojson_path, 'r') as f:
            self.geojson = json.load(f)
        
        self.class_map = self._build_class_map()
        self.geometries_by_class = self._parse_geometries()
    
    def _init_image_properties(self):
        series = self.tiff.series[0]
        
        if len(series.pages) > 1:
            self.is_pyramidal = True
            self.levels = len(series.pages)
            self.level_dimensions = []
            self.level_downsamples = []
            base_w, base_h = series.pages[0].shape[1], series.pages[0].shape[0]
            
            for i, page in enumerate(series.pages):
                h, w = page.shape[:2]
                self.level_dimensions.append((w, h))
                self.level_downsamples.append(base_w / w)
        else:
            self.is_pyramidal = False
            self.levels = 1
            page = series.pages[0]
            h, w = page.shape[:2]
            self.level_dimensions = [(w, h)]
            self.level_downsamples = [1.0]
        
        try:
            self.zarr_store = zarr.open(self.tiff.aszarr(), mode='r')
            self.use_zarr = True
        except:
            self.use_zarr = False
    
    def read_region(self, x, y, level, width, height):
        downsample = self.level_downsamples[level]
        x_level = int(x / downsample)
        y_level = int(y / downsample)
        
        if self.use_zarr and self.is_pyramidal:
            if isinstance(self.zarr_store, zarr.Group):
                level_key = list(self.zarr_store.keys())[level]
                z = self.zarr_store[level_key]
            else:
                z = self.zarr_store
            
            h_max, w_max = z.shape[-2], z.shape[-1]
            y_end = min(y_level + height, h_max)
            x_end = min(x_level + width, w_max)
            
            if len(z.shape) == 3:
                if z.shape[0] in [3, 4]:
                    region = z[:, y_level:y_end, x_level:x_end]
                    region = np.transpose(region, (1, 2, 0))
                else:
                    region = z[y_level:y_end, x_level:x_end, :]
            else:
                region = z[y_level:y_end, x_level:x_end]
            
            if region.shape[0] < height or region.shape[1] < width:
                pad_h = height - region.shape[0]
                pad_w = width - region.shape[1]
                if len(region.shape) == 3:
                    region = np.pad(region, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant')
                else:
                    region = np.pad(region, ((0, pad_h), (0, pad_w)), mode='constant')
        else:
            page = self.tiff.series[0].pages[level]
            if page.is_tiled:
                region = self._read_tiled_region(page, x_level, y_level, width, height)
            else:
                full = page.asarray()
                region = full[y_level:y_level+height, x_level:x_level+width]
        
        if len(region.shape) == 2:
            region = np.stack([region] * 3, axis=-1)
        elif region.shape[-1] == 4:
            region = region[:, :, :3]
        elif region.shape[-1] != 3:
            if region.shape[0] == 3:
                region = np.transpose(region, (1, 2, 0))
        
        return Image.fromarray(region.astype(np.uint8))
    
    def _read_tiled_region(self, page, x, y, width, height):
        tile_width = page.tilewidth
        tile_height = page.tilelength
        
        start_tile_x = x // tile_width
        start_tile_y = y // tile_height
        end_tile_x = (x + width - 1) // tile_width + 1
        end_tile_y = (y + height - 1) // tile_height + 1
        
        tiles = {}
        for ty in range(start_tile_y, end_tile_y):
            for tx in range(start_tile_x, end_tile_x):
                try:
                    tile = page.asarray(key=(ty, tx))
                    tiles[(ty, tx)] = tile
                except:
                    continue
        
        if not tiles:
            return np.zeros((height, width, 3), dtype=np.uint8)
        
        full_h = (end_tile_y - start_tile_y) * tile_height
        full_w = (end_tile_x - start_tile_x) * tile_width
        
        sample = list(tiles.values())[0]
        if len(sample.shape) == 3:
            full = np.zeros((full_h, full_w, sample.shape[-1]), dtype=sample.dtype)
        else:
            full = np.zeros((full_h, full_w), dtype=sample.dtype)
        
        for (ty, tx), tile in tiles.items():
            y_off = (ty - start_tile_y) * tile_height
            x_off = (tx - start_tile_x) * tile_width
            full[y_off:y_off+tile.shape[0], x_off:x_off+tile.shape[1]] = tile
        
        offset_y = y - start_tile_y * tile_height
        offset_x = x - start_tile_x * tile_width
        region = full[offset_y:offset_y+height, offset_x:offset_x+width]
        
        return region
    
    def _build_class_map(self):
        class_names = set()
        for feature in self.geojson.get('features', []):
            name = feature['properties'].get('name', 'unknown')
            if name:
                class_names.add(name)
        
        class_map = {'background': 0}
        for idx, name in enumerate(sorted(class_names), start=1):
            class_map[name] = idx
        return class_map
    
    def _parse_geometries(self):
        geoms_by_class = {name: [] for name in self.class_map.keys() if name != 'background'}
        
        for feature in self.geojson.get('features', []):
            props = feature['properties']
            class_name = props.get('name')
            
            if not class_name or class_name not in self.class_map:
                continue
            
            try:
                geom = shape(feature['geometry'])
                geoms_by_class[class_name].append(geom)
            except:
                continue
        
        return geoms_by_class
    
    def rasterize_tile(self, x, y, width, height, downsample):
        transform = Affine.translation(x, y) * Affine.scale(downsample, downsample)
        mask = np.zeros((height, width), dtype=np.uint8)
        
        tile_box = Polygon([
            (x, y), 
            (x + width * downsample, y),
            (x + width * downsample, y + height * downsample),
            (x, y + height * downsample)
        ])
        
        for class_name, class_id in sorted(self.class_map.items(), key=lambda x: x[1]):
            if class_name == 'background':
                continue
            
            geoms = self.geometries_by_class.get(class_name, [])
            if not geoms:
                continue
            
            intersecting = []
            for geom in geoms:
                try:
                    if geom.intersects(tile_box):
                        clipped = geom.intersection(tile_box)
                        if not clipped.is_empty:
                            intersecting.append(clipped)
                except:
                    continue
            
            if not intersecting:
                continue
            
            try:
                class_mask = features.rasterize(
                    [(g, class_id) for g in intersecting],
                    out_shape=(height, width),
                    transform=transform,
                    fill=0,
                    dtype=np.uint8,
                    default_value=class_id
                )
                mask = np.maximum(mask, class_mask)
            except:
                continue
        
        return mask
    
    def extract_tiles(self, output_dir, image_name, level=1, min_tissue_percent=10):
        """
        Extrae tiles con nombres que incluyen el nombre de la imagen original
        
        Args:
            output_dir: directorio de salida
            image_name: nombre base de la imagen (sin extensión)
            level: nivel de pirámide
            min_tissue_percent: % mínimo de tejido
        """
        os.makedirs(f"{output_dir}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/masks", exist_ok=True)
        
        width, height = self.level_dimensions[level]
        downsample = self.level_downsamples[level]
        
        step = self.tile_size - self.overlap
        y_positions = list(range(0, height - self.tile_size + 1, step))
        x_positions = list(range(0, width - self.tile_size + 1, step))
        
        saved_count = 0
        
        for y in y_positions:
            for x in x_positions:
                x0 = int(x * downsample)
                y0 = int(y * downsample)
                
                try:
                    tile_img = self.read_region(x0, y0, level, self.tile_size, self.tile_size)
                except:
                    continue
                
                tile_array = np.array(tile_img)
                gray = np.mean(tile_array, axis=2)
                tissue_percent = np.mean(gray < 240) * 100
                
                if tissue_percent < min_tissue_percent:
                    continue
                
                tile_mask = self.rasterize_tile(x0, y0, self.tile_size, self.tile_size, downsample)
                
                if np.max(tile_mask) == 0 and tissue_percent < 30:
                    continue
                
                # NOMBRE DEL TILE INCLUYE NOMBRE DE LA IMAGEN ORIGINAL
                # Formato: {nombre_imagen}_x{x}_y{y}_lvl{level}.png
                tile_name = f"{image_name}_x{x}_y{y}_lvl{level}"
                
                tile_img.save(f"{output_dir}/images/{tile_name}.png")
                Image.fromarray(tile_mask).save(f"{output_dir}/masks/{tile_name}.png")
                saved_count += 1
        
        return saved_count
    
    def close(self):
        self.tiff.close()


def process_placenta_dataset(
    tif_dir,           # Carpeta con archivos .tif
    geojson_dir,       # Carpeta con archivos .geojson
    output_dir,        # Carpeta de salida
    tile_size=512,
    overlap=256,
    level=1,
    min_tissue_percent=15
):
    """
    Procesa todas las placentas leyendo TIFs y geojsons de carpetas separadas
    
    Estructura esperada:
        tif_dir/          geojson_dir/              output_dir/
        ├── 001.tif       ├── 001.geojson           ├── images/
        ├── 002.tif       ├── 002.geojson           │   ├── 001_x0_y0_lvl1.png
        └── ...           └── ...                   │   ├── 001_x512_y0_lvl1.png
                                                    │   ├── 002_x0_y0_lvl1.png
                                                    │   └── ...
                                                    ├── masks/
                                                    │   ├── 001_x0_y0_lvl1.png
                                                    │   └── ...
                                                    └── extraction_stats.json
    """
    tif_path = Path(tif_dir)
    geojson_path = Path(geojson_dir)
    output_path = Path(output_dir)
    
    output_path.mkdir(parents=True, exist_ok=True)
    (output_path / "images").mkdir(exist_ok=True)
    (output_path / "masks").mkdir(exist_ok=True)
    
    # Encontrar todos los archivos .tif
    tif_files = sorted(tif_path.glob("*.tif")) + sorted(tif_path.glob("*.tiff"))
    
    print(f"Encontradas {len(tif_files)} imágenes en {tif_dir}")
    print(f"Buscando geojsons en {geojson_dir}")
    print("=" * 60)
    
    # Consolidar class_map global
    global_class_map = {'background': 0}
    class_id_counter = 1
    
    for tif_file in tif_files:
        # Buscar geojson correspondiente (mismo nombre, extensión .geojson o .json)
        base_name = tif_file.stem
        geojson_file = geojson_path / f"{base_name}.geojson"
        
        if not geojson_file.exists():
            geojson_file = geojson_path / f"{base_name}.json"
        
        if not geojson_file.exists():
            print(f"⚠️  No se encontró geojson para {tif_file.name}")
            continue
        
        try:
            with open(geojson_file, 'r') as f:
                geojson = json.load(f)
            
            for feature in geojson.get('features', []):
                name = feature['properties'].get('name', 'unknown')
                if name and name not in global_class_map:
                    global_class_map[name] = class_id_counter
                    class_id_counter += 1
        except Exception as e:
            print(f"Error leyendo {geojson_file}: {e}")
    
    print(f"\nMapa de clases global: {global_class_map}")
    print("=" * 60)
    
    # Procesar cada imagen
    stats = []
    
    for i, tif_file in enumerate(tif_files, 1):
        base_name = tif_file.stem
        print(f"\n[{i}/{len(tif_files)}] Procesando: {tif_file.name}")
        
        # Buscar geojson
        geojson_file = geojson_path / f"{base_name}.geojson"
        if not geojson_file.exists():
            geojson_file = geojson_path / f"{base_name}.json"
        
        if not geojson_file.exists():
            stats.append({
                'name': base_name,
                'tiles': 0,
                'status': 'ERROR: GeoJSON no encontrado'
            })
            continue
        
        try:
            extractor = QuPathTiffExtractor(
                str(tif_file),
                str(geojson_file),
                tile_size=tile_size,
                overlap=overlap
            )
            
            # Usar class_map global para consistencia entre imágenes
            extractor.class_map = global_class_map
            extractor.geometries_by_class = extractor._parse_geometries()
            
            # Extraer tiles con el nombre de la imagen
            n_tiles = extractor.extract_tiles(
                str(output_path),
                image_name=base_name,  # <-- Nombre de la imagen incluido en cada tile
                level=level,
                min_tissue_percent=min_tissue_percent
            )
            
            extractor.close()
            
            stats.append({
                'name': base_name,
                'tiles': n_tiles,
                'status': 'OK'
            })
            print(f"✅ {n_tiles} tiles extraídos")
            
        except Exception as e:
            stats.append({
                'name': base_name,
                'tiles': 0,
                'status': f'ERROR: {str(e)}'
            })
            print(f"❌ Error: {e}")
    
    # Resumen final
    print("\n" + "=" * 60)
    print("RESUMEN")
    print("=" * 60)
    total_tiles = sum(s['tiles'] for s in stats)
    ok_count = sum(1 for s in stats if s['status'] == 'OK')
    error_count = len(stats) - ok_count
    
    print(f"Total de tiles extraídos: {total_tiles}")
    print(f"Imágenes procesadas OK: {ok_count}/{len(tif_files)}")
    print(f"Imágenes con error: {error_count}")
    
    # Guardar estadísticas
    with open(output_path / 'extraction_stats.json', 'w') as f:
        json.dump({
            'class_map': global_class_map,
            'statistics': stats,
            'parameters': {
                'tile_size': tile_size,
                'overlap': overlap,
                'level': level,
                'min_tissue_percent': min_tissue_percent,
                'tif_dir': str(tif_dir),
                'geojson_dir': str(geojson_dir)
            }
        }, f, indent=2)
    
    print(f"\nEstadísticas guardadas en: {output_path / 'extraction_stats.json'}")
    
    return global_class_map, stats




In [6]:
TIF_DIRECTORY = "/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/wsi_images"           # Carpeta con archivos .tif
GEOJSON_DIRECTORY = "/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/annotations_resized" # Carpeta con archivos .geojson
OUTPUT_DIRECTORY = "/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/UNet_dataset_512x512-256vlp"        # Carpeta de salida

class_map, stats = process_placenta_dataset(
    tif_dir=TIF_DIRECTORY,
    geojson_dir=GEOJSON_DIRECTORY,
    output_dir=OUTPUT_DIRECTORY,
    tile_size=512,
    overlap=256,
    level=1,
    min_tissue_percent=15
)

Encontradas 65 imágenes en /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/wsi_images
Buscando geojsons en /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw/annotations_resized

Mapa de clases global: {'background': 0, 'Junctional Zone': 1, 'Labyrinth': 2, 'Decidua': 3, 'unknown': 4, 'Junctional': 5, 'lab2': 6, 'dec2': 7, 'jz2': 8, 'lab1': 9, 'dec1': 10, 'jz1': 11, 'dec': 12, 'lab': 13, 'jz': 14, 'Junctional zone': 15, 'Labryinth': 16, 'Decidual': 17}

[1/65] Procesando: slide_001.tif
❌ Error: list index out of range

[2/65] Procesando: slide_002.tif
❌ Error: list index out of range

[3/65] Procesando: slide_003.tif
❌ Error: list index out of range

[4/65] Procesando: slide_004.tif
❌ Error: list index out of range

[5/65] Procesando: slide_005.tif
❌ Error: list index out of range

[6/65] Procesando: slide_006.tif
❌ Error: list index out of range

[7/65] Procesando: slide_007.tif
❌ Error: list index out of range

[8/65] Procesando: slide